# Stage 2 Notebook 42 - Exp2MM Anchor + dual cls x IoU score

**Architectural fix for the cls collapse.** NB39/NB40 stuck at pred_lanes=1536 because the binary matched_existence task is structurally unstable on the 192-anchor head: cls output sigmoid is roughly uniform across all priors so the top-K decoder always picks ALL of them. Exp2LL (NB41) attacks the loss side; Exp2MM attacks the architecture.

Exp2MM keeps binary cls AND adds a parallel `iou_logits` head trained on continuous LineIoU regression. At decode time, the score becomes:

    score = sigmoid(cls_logits) * sigmoid(iou_logits)

cls answers 'is this prior a match?' (binary, with the existing ASL supervision); iou answers 'if so, how good is the geometry?' (continuous, supervised by QFL on the LineIoU target). The matching-instability that has been collapsing the binary cls task no longer determines the decode rank, because iou regression has a deterministic per-prior target that does not depend on which competing prior wins the dynamic-k contest.

Code changes (new since NB40):
- `lane_head.py CLRKDLaneHead`: `dual_score: bool` flag adds a parallel `iou_head` MLP and emits `iou_logits` (B, P).
- `losses.py FusionLossConfig`: `w_iou_aux`, `iou_aux_loss_type='qfl'`, `iou_aux_qfl_gamma=2.0`, `iou_aux_target_pow=1.0`.
- `metrics/lane_f1_decoded.py`: `score_source='cls_x_iou'` ranks priors by sigmoid(cls)*sigmoid(iou).
- `train_joint_model_experiment.py`: `eval.decoded_score_source` chooses the primary metric; when `cls_x_iou`, a `_cls_only` companion metric is also reported so we can attribute the gain.

Reference: GFLv2 (Li et al. 2021) -- predicted IoU as ranking score is the published recipe behind RTMDet's post-COCO-2022 dominance.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run -- the dual_score path is the first time the model emits `iou_logits`, so confirm the smoke test prints non-NaN losses before running 20 epochs.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. With AMP enabled, expect ~30 minutes wall-clock for 20 epochs at 3000 samples.
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp37_rmt_gca_anchor_dual_score_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp37_rmt_gca_anchor_dual_score_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp37_rmt_gca_anchor_dual_score_joint_smoke.log
OK exp37_rmt_gca_anchor_dual_score_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=6.2005 det_loss=3.6473 grad_cos=0.2734 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49851560592651367, 'gate/lane_mean': 0.5015236735343933, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp37_rmt_gca_anchor_dual_score_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp37_rmt_gca_anchor_dual_score_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp37_rmt_gca_anchor_dual_score_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp37_rmt_gca_anchor_dual_score_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp37_rmt_gca_anchor_dual_score_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp37_rmt_gca_anchor_dual_score_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp37_rmt_gca_anchor_dual_score_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/b

0

## What to watch in Exp2MM training

Reference NB40 (anchor head, ASL cls, no iou_logits): matched_iou=0.483, oracle_f1=0.418, decoded_f1=0.043, **pred_lanes=1536**.

New metrics this notebook produces:
- `val/lane/iou_aux` -- training loss of the new iou regression head (smaller is better; expect 0.4 -> 0.05 over 20 epochs).
- `val/lane/decoded_f1` -- ranks priors by sigmoid(cls) * sigmoid(iou). The primary number.
- `val/lane/decoded_cls_only_f1` -- companion metric ranking by sigmoid(cls) alone, so we can attribute the gain.
- `val/lane/decoded_oracle_f1` -- unchanged; F1 ceiling under perfect ranking.

Pass criteria at epoch 20:
- **`val/lane/decoded_f1 >= 0.20`** -- 4-5x NB40. The dual-score head should close most of the gap to oracle.
- **`val/lane/decoded_f1 - val/lane/decoded_cls_only_f1 >= 0.10`** -- the iou re-ranking is doing real work; if this gap is < 0.05, the cls head dominates and dual scoring is wasted parameters.
- **`pred_lanes < 400`** -- the iou_logits naturally have a wider score distribution so multiplying with cls produces a clear top-K cutoff.
- **`val/matched_line_iou >= 0.40`** -- preserves NB40 geometry champion.
- **`val/lane/decoded_oracle_f1 >= 0.35`** -- oracle ceiling stays high.
- `[amp] kind=bfloat16 enabled=True` log line at start.

Failure signals:
- iou_aux loss does not decrease: iou head is decoupled from the geometry it should describe. Bump `w_iou_aux` to 3.0.
- decoded_f1 ~ decoded_cls_only_f1: the iou head learns to mirror cls. Try `iou_aux_target_pow: 0.5` to expand low-IoU range.
- matched_iou drops below 0.30: the dual_score MLP is competing with offset_head/param_head for embed_dim capacity. Increase embed_dim 128 -> 192 or set `cls_separate_path: true` to give cls + iou their own ROI features.